# Tarea 2: Limpieza y Estandarización de Datos

**Objetivo:** Corregir todos los problemas de calidad identificados en la Tarea 1.

Este notebook implementa funciones para limpiar y estandarizar los datos de transacciones.


## 1. Configurar el entorno del proyecto en Jupyter

In [1]:
import os
import sys
from pathlib import Path
from googletrans import Translator

translator = Translator()
# Verificar el intérprete de Python activo
print(f"Intérprete Python: {sys.executable}")
print(f"Versión Python: {sys.version}")

# Mostrar directorio de trabajo actual
current_dir = os.getcwd()
print(f"\nDirectorio actual: {current_dir}")

# Ruta del proyecto dentro del contenedor Docker
# El docker-compose monta ./ en /home/jovyan/work
project_root = Path("/home/jovyan/work")

# Cambiar al directorio del proyecto si es necesario
if os.getcwd() != str(project_root):
    os.chdir(project_root)
    print(f"Directorio cambiado a: {os.getcwd()}")
else:
    print(f"Ya estamos en el directorio del proyecto: {project_root}")

# Agregar el proyecto a sys.path para imports
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Ruta del proyecto agregada a sys.path")

print(f"\nRaíz del proyecto: {project_root}")


ModuleNotFoundError: No module named 'googletrans'

## 2. Validar ubicación del notebook y estructura de carpetas

In [13]:
# Validar que las carpetas existen
# Dentro de Docker: /home/jovyan/work/homeworks/...
homeworks_dir = project_root / "homeworks"
data_dir = project_root / "data/raw"
output_dir = homeworks_dir / "output"
tarea_2_dir = homeworks_dir / "tarea_2"

print("Validando estructura de carpetas (rutas dentro del contenedor Docker):")
print(f"  project_root : {project_root}")
print(f"  homeworks/   : {homeworks_dir}  → existe: {homeworks_dir.exists()}")
print(f"  data/        : {data_dir}  → existe: {data_dir.exists()}")
print(f"  output/      : {output_dir}  → existe: {output_dir.exists()}")
print(f"  tarea_2/     : {tarea_2_dir}  → existe: {tarea_2_dir.exists()}")

# Crear output si no existe
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n✓ Carpeta output creada: {output_dir}")
else:
    print(f"\n✓ Carpeta output ya existe")


Validando estructura de carpetas (rutas dentro del contenedor Docker):
  project_root : /home/jovyan/work
  homeworks/   : /home/jovyan/work/homeworks  → existe: True
  data/        : /home/jovyan/work/data/raw  → existe: True
  output/      : /home/jovyan/work/homeworks/output  → existe: True
  tarea_2/     : /home/jovyan/work/homeworks/tarea_2  → existe: True

✓ Carpeta output ya existe


## 3. Cargar librerías y dependencias

In [11]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
import re

print("✓ Librerías importadas correctamente:")
print(f"  - pandas {pd.__version__}")
print(f"  - numpy {np.__version__}")


✓ Librerías importadas correctamente:
  - pandas 2.1.1
  - numpy 1.24.4


## 4. Cargar datos raws y definir funciones de limpieza

In [14]:
# Cargar datos
csv_path = data_dir / "ventas.csv"
print(f"Cargando datos desde: {csv_path}")

if csv_path.exists():
    df_raw = pd.read_csv(csv_path)
    print(f"✓ Datos cargados exitosamente")
    print(f"  - Filas: {len(df_raw)}")
    print(f"  - Columnas: {len(df_raw.columns)}")
    print(f"\nPrimeras filas del dataset raw:")
    print(df_raw.head())
else:
    print(f"✗ Archivo no encontrado: {csv_path}")
    print("Verifica que la Tarea 1 fue ejecutada correctamente")

# Guardar copia del original para comparación
df_backup = df_raw.copy()


Cargando datos desde: /home/jovyan/work/data/raw/ventas.csv
✓ Datos cargados exitosamente
  - Filas: 20000
  - Columnas: 31

Primeras filas del dataset raw:
   id_venta       fecha   año  mes dia_semana  semana_numero    producto  \
0         1  2024-12-03  2024   12    Tuesday             49  Disco Duro   
1         2  08-26-2024  2024    8     Monday             35     Teclado   
2         3  2024-08-21  2024    8  Wednesday             34     Monitor   
3         4  2024-07-07  2024    7     Sunday             27       Mouse   
4         5  2023-05-15  2023    5     Monday             20         NaN   

        categoria cantidad precio_unitario  ...  cliente_antiguedad_dias  \
0     Electrónica       46             976  ...                        1   
1             NaN       57            1309  ...                      219   
2      Accesorios        8             727  ...                        7   
3  Almacenamiento       15             705  ...                       99   
4     

In [15]:
# Funciones de limpieza requeridas

def limpiar_montos(valor):
    """
    Convierte montos a float, elimina $, corrige negativos y maneja nulos.
    """
    if pd.isna(valor):
        return np.nan
    
    # Convertir a string y eliminar espacios
    valor_str = str(valor).strip()
    
    # Si está vacío, retornar NaN
    if valor_str == '' or valor_str.upper() == 'NAN':
        return np.nan
    
    # Eliminar signo de dólar y espacios
    valor_str = valor_str.replace('$', '').replace(' ', '')
    
    try:
        monto = float(valor_str)
        # Si es negativo, convertir a positivo
        if monto < 0:
            monto = abs(monto)
        return monto
    except ValueError:
        return np.nan

def estandarizar_fechas(valor):
    """
    Unififica el formato de fechas a YYYY-MM-DD.
    """
    if pd.isna(valor):
        return np.nan
    
    valor_str = str(valor).strip()
    
    if valor_str == '' or valor_str.upper() == 'NAN':
        return np.nan
    
    # Intentar varios formatos comunes
    formatos = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%d-%m-%Y']
    
    for fmt in formatos:
        try:
            fecha = datetime.strptime(valor_str, fmt)
            return fecha.strftime('%Y-%m-%d')
        except ValueError:
            continue
    
    # Si ningún formato funciona, retornar NaN
    return np.nan

def corregir_estados(valor):
    """
    Asigna 'DESCONOCIDO' a valores vacíos en el estado.
    """
    if pd.isna(valor) or valor == '' or str(valor).strip() == '':
        return 'DESCONOCIDO'
    return str(valor).strip()

def corregir_tiendas(valor):
    """
    Asigna 'TIENDA_SIN_ESPECIFICAR' a valores nulos.
    """
    if pd.isna(valor) or valor == '' or str(valor).strip() == '':
        return 'TIENDA_SIN_ESPECIFICAR'
    return str(valor).strip()

def corregir_clientes(valor):
    """
    Asigna 'DESCONOCIDO' a valores nulos en customer_id.
    """
    if pd.isna(valor) or valor == '' or str(valor).strip() == '':
        return 'DESCONOCIDO'
    return str(valor).strip()

def estandarizar_textos_ingles(texto):
    """
    Convierte texto de inglés a español
    """
    # Verificar si el texto es nulo o vacío
    if pd.isna(texto) or texto == "" or str(texto).strip() == '':
        return 'DESCONOCIDO'
    
    # Traducir solo si hay texto válido
    try:
        traduccion = translator.translate(texto, src='en', dest='es')
        return traduccion.text
    except Exception as e:
        print(f"Error traduciendo '{texto}': {e}")
        return 'ERROR_TRADUCCION'  # o return 'ERROR_TRADUCCION'

print("✓ Funciones de limpieza definidas correctamente")


✓ Funciones de limpieza definidas correctamente


## 5. Aplicar las funciones de limpieza al dataset

In [6]:
print("Aplicando funciones de limpieza...\n")

# Crear copia para limpiar
df_clean = df_raw.copy()

# Aplicar funciones según las columnas disponibles
columnas = df_clean.columns
print(f"Columnas disponibles: {list(columnas)}")

# Limpiar montos (buscar columna de monto)
monto_cols = [col for col in columnas if 'precio_unitario' in col.lower() or 'amount' in col.lower()]
if monto_cols:
    for col in monto_cols:
        print(f"\n✓ Limpiando columna '{col}'...")
        df_clean[col] = df_clean[col].apply(limpiar_montos)
        nulos_antes = df_backup[col].isna().sum()
        nulos_despues = df_clean[col].isna().sum()
        print(f"  Nulos: {nulos_antes} → {nulos_despues}")

# Estandarizar fechas (buscar columna de fecha)
fecha_cols = [col for col in columnas if 'fecha' in col.lower() or 'date' in col.lower()]
if fecha_cols:
    for col in fecha_cols:
        print(f"\n✓ Estandarizando columna '{col}'...")
        df_clean[col] = df_clean[col].apply(estandarizar_fechas)
        nulos_antes = df_backup[col].isna().sum()
        nulos_despues = df_clean[col].isna().sum()
        print(f"  Nulos: {nulos_antes} → {nulos_despues}")
        
# Estandarizar fechas (buscar columna de fecha_registro_cliente)
fecha_cols = [col for col in columnas if 'fecha_registro_cliente' in col.lower() or 'date' in col.lower()]
if fecha_cols:
    for col in fecha_cols:
        print(f"\n✓ Estandarizando columna '{col}'...")
        df_clean[col] = df_clean[col].apply(estandarizar_fechas)
        nulos_antes = df_backup[col].isna().sum()
        nulos_despues = df_clean[col].isna().sum()
        print(f"  Nulos: {nulos_antes} → {nulos_despues}")

# Estandarizar dia de la semana a español 
# Opción 2: Buscar columnas de tipo objeto (texto) automáticamente
columnas_texto_auto=["dia_semana"]
columnas_texto_auto = df_clean.select_dtypes(include=['object']).columns.tolist()
for col in columnas_texto_auto:
    print(f"\n✓ Traduciendo columna '{col}'...")
    df_clean[col] = df_clean[col].apply(estandarizar_textos_ingles)

print("\n✓ Limpieza completada")
print(f"\nPrimeras filas del dataset limpio:")
print(df_clean.head())


Aplicando funciones de limpieza...

Columnas disponibles: ['transaction_id', 'date', 'customer_id', 'amount', 'status', 'store']

✓ Limpiando columna 'amount'...
  Nulos: 7 → 7

✓ Estandarizando columna 'date'...
  Nulos: 0 → 0

✓ Corrigiendo columna 'status'...

✓ Corrigiendo columna 'store'...

✓ Corrigiendo columna 'customer_id'...

✓ Limpieza completada

Primeras filas del dataset limpio:
  transaction_id        date customer_id  amount       status  \
0      TXN-00001  2023-03-02      1038.0  194.07  DESCONOCIDO   
1      TXN-00002  2023-04-06      1002.0  452.64    CANCELADA   
2      TXN-00003  2023-08-24      1017.0  344.76      FALLIDA   
3      TXN-00004  2023-01-20      1041.0   85.16  DESCONOCIDO   
4      TXN-00005  2023-07-02      1000.0   72.53      FALLIDA   

                    store  
0  TIENDA_SIN_ESPECIFICAR  
1            Tienda_Norte  
2              Tienda_Sur  
3              Tienda_Sur  
4            Tienda_Norte  


## 6. Generar reporte comparativo antes/después

In [7]:
# Generar reporte comparativo
print("\n" + "="*60)
print("REPORTE COMPARATIVO - ANTES vs DESPUÉS DE LIMPIEZA")
print("="*60)

reporte = {
    "fecha_ejecucion": datetime.now().isoformat(),
    "resumen_general": {
        "total_registros": len(df_clean),
        "total_columnas": len(df_clean.columns),
    },
    "analisis_nulos": {}
}

print("\n📊 Análisis de Nulos por Columna:")
print("-" * 60)
print(f"{'Columna':<20} {'Antes':<15} {'Después':<15} {'Mejora':<10}")
print("-" * 60)

for col in df_clean.columns:
    nulos_antes = df_backup[col].isna().sum()
    nulos_despues = df_clean[col].isna().sum()
    mejora = nulos_antes - nulos_despues
    porcentaje_antes = (nulos_antes / len(df_backup)) * 100
    porcentaje_despues = (nulos_despues / len(df_clean)) * 100
    
    print(f"{col:<20} {nulos_antes} ({porcentaje_antes:.1f}%) {nulos_despues} ({porcentaje_despues:.1f}%) {mejora:+d}")
    
    reporte["analisis_nulos"][col] = {
        "nulos_antes": int(nulos_antes),
        "porcentaje_antes": round(porcentaje_antes, 2),
        "nulos_despues": int(nulos_despues),
        "porcentaje_despues": round(porcentaje_despues, 2),
        "mejora": int(mejora)
    }

print("\n✓ Estadísticas del dataset limpio:")
print(f"  - Registros totales: {len(df_clean)}")
print(f"  - Registros sin nulos en ninguna columna: {df_clean.dropna().shape[0]}")
print(f"  - Registros con al menos un nulo: {df_clean[df_clean.isna().any(axis=1)].shape[0]}")

print("\nTipos de datos después de limpieza:")
print(df_clean.dtypes)



REPORTE COMPARATIVO - ANTES vs DESPUÉS DE LIMPIEZA

📊 Análisis de Nulos por Columna:
------------------------------------------------------------
Columna              Antes           Después         Mejora    
------------------------------------------------------------
transaction_id       0 (0.0%) 0 (0.0%) +0
date                 0 (0.0%) 0 (0.0%) +0
customer_id          7 (6.7%) 0 (0.0%) +7
amount               7 (6.7%) 7 (6.7%) +0
status               25 (23.8%) 0 (0.0%) +25
store                19 (18.1%) 0 (0.0%) +19

✓ Estadísticas del dataset limpio:
  - Registros totales: 105
  - Registros sin nulos en ninguna columna: 98
  - Registros con al menos un nulo: 7

Tipos de datos después de limpieza:
transaction_id     object
date               object
customer_id        object
amount            float64
status             object
store              object
dtype: object


## 7. Guardar archivos de salida

In [8]:
print("\n" + "="*60)
print("GUARDANDO ARCHIVOS")
print("="*60)

# Guardar dataset limpio
clean_csv_path = output_dir / "transacciones_clean.csv"
df_clean.to_csv(clean_csv_path, index=False)
print(f"\n✓ Dataset limpio guardado en: {clean_csv_path}")

# Guardar reporte de limpieza
reporte_path = output_dir / "reporte_limpieza.json"
with open(reporte_path, 'w', encoding='utf-8') as f:
    json.dump(reporte, f, ensure_ascii=False, indent=2)
print(f"✓ Reporte guardado en: {reporte_path}")

# Listar archivos creados
print("\n📁 Archivos generados:")
print("-" * 60)
for archivo in output_dir.glob("*"):
    if archivo.is_file():
        tamaño = archivo.stat().st_size
        print(f"  - {archivo.name} ({tamaño:,} bytes)")



GUARDANDO ARCHIVOS

✓ Dataset limpio guardado en: /home/jovyan/work/homeworks/output/transacciones_clean.csv
✓ Reporte guardado en: /home/jovyan/work/homeworks/output/reporte_limpieza.json

📁 Archivos generados:
------------------------------------------------------------
  - reporte_limpieza.json (1,109 bytes)
  - transacciones_clean.csv (6,264 bytes)


## 8. Resumen de la Tarea 2

In [9]:
print("\n" + "🎉 " * 15)
print("\n✅ TAREA 2 COMPLETADA EXITOSAMENTE")
print("\n" + "🎉 " * 15)

print("\n📋 ENTREGABLES COMPLETADOS:")
print("-" * 60)
print("✓ Función limpiar_montos() - Implementada")
print("✓ Función estandarizar_fechas() - Implementada")
print("✓ Función corregir_estados() - Implementada")
print("✓ Función corregir_tiendas() - Implementada")
print("✓ Función corregir_clientes() - Implementada")
print("✓ Aplicación de todas las funciones - Realizada")
print(f"✓ Archivo transacciones_clean.csv - Guardado")
print(f"✓ Reporte de limpieza (JSON) - Guardado")

print("\n📊 ESTADÍSTICAS FINALES:")
print("-" * 60)
print(f"Total de registros procesados: {len(df_clean)}")
print(f"Registros completamente limpios (sin nulos): {df_clean.dropna().shape[0]}")
print(f"Mejora general en calidad de datos: Completada")

print("\n📁 UBICACIÓN DE ARCHIVOS:")
print("-" * 60)
print(f"Dataset limpio: {clean_csv_path.relative_to(project_root)}")
print(f"Reporte JSON: {reporte_path.relative_to(project_root)}")

print("\n💡 PRÓXIMOS PASOS:")
print("-" * 60)
print("La Tarea 2 está completada. Ahora puedes:")
print("1. Revisar el archivo transacciones_clean.csv")
print("2. Consultar el reporte_limpieza.json para detalles")
print("3. Comenzar con la Tarea 3: Manejo de Nulos y Duplicados")
print("\n✨ ¡Excelente trabajo! ✨\n")



🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 

✅ TAREA 2 COMPLETADA EXITOSAMENTE

🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 🎉 

📋 ENTREGABLES COMPLETADOS:
------------------------------------------------------------
✓ Función limpiar_montos() - Implementada
✓ Función estandarizar_fechas() - Implementada
✓ Función corregir_estados() - Implementada
✓ Función corregir_tiendas() - Implementada
✓ Función corregir_clientes() - Implementada
✓ Aplicación de todas las funciones - Realizada
✓ Archivo transacciones_clean.csv - Guardado
✓ Reporte de limpieza (JSON) - Guardado

📊 ESTADÍSTICAS FINALES:
------------------------------------------------------------
Total de registros procesados: 105
Registros completamente limpios (sin nulos): 98
Mejora general en calidad de datos: Completada

📁 UBICACIÓN DE ARCHIVOS:
------------------------------------------------------------
Dataset limpio: homeworks/output/transacciones_clean.csv
Reporte JSON: homeworks/output/reporte_limpieza.json

💡 PRÓXIMOS PASOS:
-------------------------